<a href="https://colab.research.google.com/github/SukhjeetxSingh/ReACT_based_AI_fraud_detection/blob/main/ReACT_based_AI_fraud_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
from openai import OpenAI
import sys
import os
from google.colab import userdata
from IPython.display import Markdown, display
import json

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
import os
import sys
from google.colab import userdata

# Retrieve your secret path from the Colab secrets manager
PROJECT_ROOT = userdata.get('PROJECT_ROOT')

# Add it to the system path if it's not already there
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"Project root successfully loaded from secrets: {PROJECT_ROOT}")

Project root successfully loaded from secrets: /content/drive/MyDrive/Prompting for LLM Reasoning and Planning for Financial Services/Module 4 Exercise/Fraud_Detection_Agent


In [32]:
# Import only what you need
from tools.investigation_tools import (
    get_transaction_history,
    get_customer_profile,
    check_regulatory_thresholds,
    process_tool_calls
)

In [33]:
import importlib
import tools.investigation_tools as inv

# This command reloads the updated file from your Drive into memory
importlib.reload(inv)

# Now access your functions through the 'inv' alias
# Example: inv.get_transaction_history(...)
print("Module reloaded and ready to use!")

Module reloaded and ready to use!


In [34]:
# Securely fetch the API key using your specific secret name
api_key = userdata.get('Udacity_Vocareum_OpenAI_Key')

client = OpenAI(
    base_url="https://openai.vocareum.com/v1",
    api_key=api_key
)

print("OpenAI client initialized with Vocareum base URL using Udacity secret key.")


OpenAI client initialized with Vocareum base URL using Udacity secret key.


## Quick Tool Demo: See What We're Working With

In [35]:
# TODO: Quick demo of our investigation tools
# Test each tool to understand what data they return

print("🔍 TOOL DEMO - Transaction History:")
# TODO: Call get_transaction_history with account "high_risk_account_001" for 14 days
demo_transactions = get_transaction_history("high_risk_account_001")
print(json.dumps(demo_transactions, indent=2))

print("\n👤 TOOL DEMO - Customer Profile:")
# TODO: Call get_customer_profile for "CUST_001"
demo_customer = get_customer_profile("CUST_001")
print(json.dumps(demo_customer, indent=2))

print("\n📋 TOOL DEMO - Regulatory Check:")
# TODO: Call check_regulatory_thresholds for amount 9800, type "cash_deposit"
demo_check = check_regulatory_thresholds(9800, "cash_deposit")
print(json.dumps(demo_check, indent=2))

🔍 TOOL DEMO - Transaction History:
{
  "account_id": "high_risk_account_001",
  "period_days": 30,
  "transaction_count": 6,
  "transactions": [
    {
      "date": "18-05-2026",
      "amount": 9860,
      "type": "cash_deposit",
      "location": "branch_A"
    },
    {
      "date": "17-05-2026",
      "amount": 9800,
      "type": "cash_deposit",
      "location": "branch_B"
    },
    {
      "date": "16-05-2026",
      "amount": 9560,
      "type": "cash_deposit",
      "location": "branch_A"
    },
    {
      "date": "15-05-2026",
      "amount": 9990,
      "type": "cash_deposit",
      "location": "branch_C"
    },
    {
      "date": "14-05-2026",
      "amount": 8860,
      "type": "cash_deposit",
      "location": "branch_A"
    },
    {
      "date": "13-05-2026",
      "amount": 8910,
      "type": "cash_deposit",
      "location": "branch_A"
    }
  ]
}

👤 TOOL DEMO - Customer Profile:
{
  "name": "Maria Santos",
  "occupation": "Restaurant Manager",
  "annual_income": 

In [36]:
# 1. Reload the module to pick up your latest changes
importlib.reload(inv)

print("🔍 TOOL DEMO - Transaction History:")
# 2. Use the 'inv.' prefix instead of the raw function name
demo_transactions = inv.get_transaction_history("high_risk_account_001")
print(json.dumps(demo_transactions, indent=2))

print("\n👤 TOOL DEMO - Customer Profile:")
demo_customer = inv.get_customer_profile("CUST_001")
print(json.dumps(demo_customer, indent=2))

print("\n📋 TOOL DEMO - Regulatory Check:")
demo_check = inv.check_regulatory_thresholds(9800, "cash_deposit")
print(json.dumps(demo_check, indent=2))

🔍 TOOL DEMO - Transaction History:
{
  "account_id": "high_risk_account_001",
  "period_days": 30,
  "transaction_count": 6,
  "transactions": [
    {
      "date": "18-05-2026",
      "amount": 9860,
      "type": "cash_deposit",
      "location": "branch_A"
    },
    {
      "date": "17-05-2026",
      "amount": 9800,
      "type": "cash_deposit",
      "location": "branch_B"
    },
    {
      "date": "16-05-2026",
      "amount": 9560,
      "type": "cash_deposit",
      "location": "branch_A"
    },
    {
      "date": "15-05-2026",
      "amount": 9990,
      "type": "cash_deposit",
      "location": "branch_C"
    },
    {
      "date": "14-05-2026",
      "amount": 8860,
      "type": "cash_deposit",
      "location": "branch_A"
    },
    {
      "date": "13-05-2026",
      "amount": 8910,
      "type": "cash_deposit",
      "location": "branch_A"
    }
  ]
}

👤 TOOL DEMO - Customer Profile:
{
  "name": "Maria Santos",
  "occupation": "Restaurant Manager",
  "annual_income": 

## ReACT Prompt: Teaching LLM to Use Tools

In [37]:
# TODO: Create ReACT prompt that teaches the LLM to use our tools
REACT_PROMPT = """
You are a Financial Crimes Investigator using real investigation tools.

AVAILABLE TOOLS:
# TODO: Add tool descriptions here
1. get_transaction_history(account_id, days) - Get account transactions
2. get_customer_profile(customer_id) - Get customer info and risk data
3. check_regulatory_thresholds(amount, type) - Check compliance requirements

PROCESS: Use THOUGHT → ACTION → OBSERVATION cycle:

THOUGHT: [Decide what you need to investigate]
ACTION: [Call tools using this exact JSON format]
```json
{
  "tool": "tool_name",
  "parameters": {"param1": "value1", "param2": "value2"}
}
```
OBSERVATION: [Analyze the tool results]

Continue until you have enough information for a final recommendation.
"""

print("📋 ReACT prompt configured for tool usage")

📋 ReACT prompt configured for tool usage


## Investigation Scenario: Suspicious Cash Deposits

In [38]:
# Investigation case for ReACT analysis
investigation_case = """
SUSPICIOUS ACTIVITY ALERT

Case: Multiple cash deposits just under $10,000
Customer ID: CUST_001
Account ID: high_risk_account_001
Time Period: Past 14 days
Alert Source: Branch manager noticed pattern
Customer Explanation: "Restaurant business doing really well"

TASK: Investigate and determine if SAR filing is required.
"""

print("📁 Investigation scenario loaded")
print("Focus: Potential cash structuring pattern")

📁 Investigation scenario loaded
Focus: Potential cash structuring pattern


## ReACT Investigation in Action

In [39]:
importlib.reload(inv)


<module 'tools.investigation_tools' from '/content/drive/MyDrive/Prompting for LLM Reasoning and Planning for Financial Services/Module 4 Exercise/Fraud_Detection_Agent/tools/investigation_tools.py'>

In [43]:
def run_react_investigation(case_details, max_rounds=3):
    """Run ReACT investigation with real tool integration"""

    print("🚀 STARTING ReACT INVESTIGATION")
    print("=" * 50)

    context = case_details

    for round_num in range(1, max_rounds + 1):
        print(f"\n🔄 ROUND {round_num}")
        print("-" * 30)

        # TODO: Create prompt combining REACT_PROMPT with case details
        prompt = f"""{REACT_PROMPT}

CASE DETAILS:
{context}

Conduct your investigation using THOUGHT → ACTION → OBSERVATION.
"""

        try:
            # TODO: Get LLM reasoning and tool calls
            response = client.chat.completions.create(
                model= "gpt-4o-mini",  # e.g., "gpt-4o"
                messages=[{"role":"user", "content": prompt}],  # Construct messages with system and user roles
                temperature=0.3,
                max_tokens=600
            )

            llm_response = response.choices[0].message.content  # Extract LLM text response
            print("🤖 INVESTIGATOR RESPONSE:")
            print(llm_response)
            print()


            # TODO: Execute any tool calls using process_tool_calls
            tool_results = process_tool_calls(llm_response)  # Parse and execute tool calls from llm_response
            # TODO: Handle tool results and update context
            if tool_results:
                # Add tool results to context for next round
                context += f"\n\nROUND {round_num} TOOL RESULTS:\n"
                for result in tool_results:
                    context += f"- {result['tool']}: {json.dumps(result['result'], indent=2)}\n"
            else:
                # No tools called - likely final conclusion
                print("✅ Investigation complete - no more tools needed")
                break

        except Exception as e:
            print(f"❌ Error in Round {round_num}: {str(e)}")
            break

    print("\n" + "=" * 50)
    print("🏁 INVESTIGATION COMPLETE")

# TODO: Run the investigation once you've implemented the function
run_react_investigation(investigation_case)
print("ReACT investigation function ready to implement")

🚀 STARTING ReACT INVESTIGATION

🔄 ROUND 1
------------------------------
🤖 INVESTIGATOR RESPONSE:
THOUGHT: I need to investigate the transaction history of the account to understand the pattern of cash deposits just under $10,000. This will help me determine if the activity is suspicious and if a Suspicious Activity Report (SAR) filing is required.

ACTION:
```json
{
  "tool": "get_transaction_history",
  "parameters": {"account_id": "high_risk_account_001", "days": 14}
}
```

OBSERVATION: [I will analyze the transaction history results once received.]

---

THOUGHT: After reviewing the transaction history, I need to gather more information about the customer to assess their risk profile and business operations.

ACTION:
```json
{
  "tool": "get_customer_profile",
  "parameters": {"customer_id": "CUST_001"}
}
```

OBSERVATION: [I will analyze the customer profile results once received.]

---

THOUGHT: Depending on the amounts and types of transactions observed, I may need to check if a

## Compare: Basic vs ReACT with Tools

Let's see the difference between basic analysis and ReACT with real tools:

In [41]:
# TODO: Compare basic analysis without tools vs ReACT with tools
print("📊 COMPARISON: Basic Analysis vs ReACT with Tools\n")

print("🔸 BASIC ANALYSIS (No Tools):")
basic_prompt = "Analyse this suspicious activity case and provide recommendation" #TODO: Create a simple prompt for basic analysis

# TODO: Get basic response from OpenAI without tools
basic_response = client.chat.completions.create(
    model= "gpt-4o-mini",
    messages=[{"role":"user", "content":f"{basic_prompt}\n\n{investigation_case}"}],
    temperature = 0.3,
    max_tokens = 300
)

# print(basic_response.choices[0].message.content)
print("TODO: Implement basic analysis call")

print("\n" + "="*60)
print("🔸 ReACT WITH TOOLS (Above):")
print("- Used real transaction data")
print("- Checked actual customer profile")
print("- Verified regulatory thresholds")
print("- Made data-driven decisions")
print("\n✅ ReACT with tools provides evidence-based investigation!")

📊 COMPARISON: Basic Analysis vs ReACT with Tools

🔸 BASIC ANALYSIS (No Tools):
TODO: Implement basic analysis call

🔸 ReACT WITH TOOLS (Above):
- Used real transaction data
- Checked actual customer profile
- Verified regulatory thresholds
- Made data-driven decisions

✅ ReACT with tools provides evidence-based investigation!


## Test Your Own Scenario

In [42]:
# Template for testing different scenarios

custom_scenario = """
INVESTIGATION CASE: SUSPICIOUS ACTIVITY ALERT

Case: Multiple unusal transactions
Customer ID: CUST_002
Account ID: normal_account
Time Period: Past 30 days
Alert Source: Branch manager noticed pattern
Customer Explanation: "Recently recieved a great compensation"


TASK: Investigate and determine if SAR or other kind of filing is required.
"""

print("💡 TESTING FRAMEWORK:")
print("1. Fill in the template above")
print("2. Run: run_react_investigation(your_scenario)")

run_react_investigation(custom_scenario, max_rounds=3 )

print("3. Watch ReACT use tools to investigate")
print("\nAvailable test combinations:")
print("- CUST_001 + high_risk_account_001: Restaurant manager, cash deposits")
print("- CUST_002 + business_account_002: Business owner, wire transfers")
print("- CUST_003 + normal_account: Software engineer, normal transactions")

# TODO: Try creating your own scenario and testing it!

💡 TESTING FRAMEWORK:
1. Fill in the template above
2. Run: run_react_investigation(your_scenario)
🚀 STARTING ReACT INVESTIGATION

🔄 ROUND 1
------------------------------
🤖 INVESTIGATOR RESPONSE:
THOUGHT: I need to investigate the transaction history of the account to identify the unusual transactions that triggered the alert. I will also gather the customer profile to understand the customer's risk level and background.

ACTION: First, I will retrieve the transaction history for the account over the past 30 days.

```json
{
  "tool": "get_transaction_history",
  "parameters": {"account_id": "normal_account", "days": 30}
}
```

OBSERVATION: [I will analyze the transaction history once I receive the results.]


Executing: get_transaction_history

Parameters: {'account_id': 'normal_account', 'days': 30}
Result: [
  {
    "tool": "get_transaction_history",
    "parameters": {
      "account_id": "normal_account",
      "days": 30
    },
    "result": {
      "account_id": "normal_account"